# 8 — Training loop

**Before:** notebook **7** (batches + loss).

**This notebook:** `Trainer` — same structure as llm.c / `train_gpt2.py`.

Set `RUN_TRAIN = True` to run 300 steps (~seconds on tiny GPT). Default is a quick 50-step demo.

**Dojo (optional):** `dojo-grade --lesson C2-L08`


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text, train_val_split
from llmc.model import GPT, GPTConfig
from llmc.train import Trainer, TrainConfig

text = load_text(DATA)
train_text, val_text = train_val_split(text)
tok = CharTokenizer.from_text(text)
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)

config = GPTConfig.tiny(vocab_size=tok.vocab_size, block_size=64)
model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"

RUN_TRAIN = False  # True → 300 steps; False → 50-step quick demo
max_steps = 300 if RUN_TRAIN else 50

trainer = Trainer(
    model,
    train_ids,
    val_ids,
    TrainConfig(max_steps=max_steps, batch_size=32, eval_interval=25, learning_rate=3e-3),
    device=device,
)
print(f"device={device}, max_steps={max_steps}")


In [ ]:
if RUN_TRAIN:
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    history = trainer.train()
    for row in history:
        print(f"step {row['step']:4d} | train {row['train']:.4f} | val {row['val']:.4f}")
    torch.save({"model": trainer.model.state_dict(), "config": config}, CHECKPOINT)
    print("saved", CHECKPOINT)
else:
    print("Training skipped — set RUN_TRAIN=True (trainer configured above with max_steps)")


In [ ]:
# --- Fast verify (seconds) ---
import subprocess

tests = ROOT / "tests" / "test_train.py"
if tests.is_file():
    cmd = [sys.executable, "-m", "pytest", str(tests), "-q", "-k", "trainer_short"]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
    print("pytest OK — ready for dojo-grade --lesson C2-L08")
else:
    print("Skip pytest:", tests)
